In [1]:
! pip install python-terrier[all]

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
import pandas as pd
import re
import gdown
import path
import requests
import os
import subprocess
import pyterrier as pt
import numpy as np
import os
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch
from transformers import AutoModelForSequenceClassification
from torch.nn import BCEWithLogitsLoss
from torch.optim import AdamW
from matplotlib import pyplot as plt

if not pt.started():
    pt.init()

/tmp/ipykernel_78406/68132618.py:10: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_78406/68132618.py:11: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
download_mantis= False
download_trec = False
download_QQP = False


In [4]:
#download mantis
path_mantis = "dataset/mantis"

if download_mantis:
    os.makedirs(path_mantis,exist_ok=True)
    # Download by file ID
    gdown.download(id="17Uj9EwyGGCk9w_LIqDjlTx1y4MU7xxPv", output="mantis.zip")
    # à unzip dans le dossier mantis


# download TREC 2020
if download_trec:
    url_base_trec = "https://msmarco.z22.web.core.windows.net/msmarcoranking"
    
    files_trec = [
        "collection.tar.gz", 
        "queries.tar.gz",
        "qrels.dev.tsv",
        "qrels.train.tsv"
    ]
    
    output_path_trec = "dataset/trec"
    os.makedirs(output_path_trec, exist_ok=True)
    
    for file in files_trec:
        url = f"{url_base_trec}/{file}"
        
        cmd = [
            "/users/Etu9/21510119/.conda/envs/env_ri/bin/aria2c",
            "-x", "16",          
            "-s", "16",          
            "-k", "1M",          
            "-d", output_path_trec,
            "-o", file,
            url
        ]
        
        print(f"Téléchargement de {file}...")
        subprocess.run(cmd, check=True)
        
        print(f"{file} téléchargé !")

# download QQP (Quora Question Pairs)


# Mantis

In [5]:
path_mantis = "dataset/mantis"

mantis_test = pd.read_csv(f"{path_mantis}/test.tsv", sep="\t")
mantis_train = pd.read_csv(f"{path_mantis}/train.tsv", sep="\t")
mantis_valid = pd.read_csv(f"{path_mantis}/valid.tsv", sep="\t")

In [6]:
mantis_train

,context,response
0,"""I have a mid-2007 MacBook that, according to ...",No idea. I think that thread had a bit of a di...
1,The iPhone 3G supports Bluetooth stereo (A2DP)...,Yes we both have iPhones. I think the sound qu...
2,"""I keep hearing all this hoopla about BBM (Bla...",I believe 2 or 3 of the ones I listed above AL...
3,I was a very happy customer of my microcell un...,It sounds like it's directly related to me. Ne...
4,I’ve just started using iTunes U on my iPhone ...,"""I have """"iTunes U"""" hidden under the General ..."
...,...,...
82267,"""I'm working on creating a species that favour...","""@Century I just plugged *""""minimum viable pop..."
82268,In my desert canal world all water is supplied...,I fixed the numbers which I had obviously gott...
82269,"""I'm in the process of designing a creature lo...",I am having trouble visualizing what you descr...
82270,"""There is a walled town, situated at a river f...","@N.Presley, I mentioned Cossacks, because your..."


In [7]:
def process_mantis(df):
    """
    processing du dataset mantis
    Args:
        df (pd.DataFrame) : dataframe contenant deux colonnes, context et response, context correspondant à la query et response au résultat (l'element pertinent)
    Returns:
        queries (pd.DataFrame) : dataframe résultant du processing, pour l'instant on ajoute juste une colonne label pour indiquer si le document est pertinent
        corpus (pd.DataFrame) : dataframe
        label
        index
    """
    # création des colonnes manquantes
    df["label"] =1
    df['qid'] = "q" + (df.index + 1).astype(str)
    df['docno'] = "d" + (df.index + 1).astype(str)

    # renommage des colonnes pour la compatibilité avec pyterrier
    df = df.rename(columns={
        "context" : "query",
        "response" : "text"
    })

    #slit - un df : qid, query - un df : qid, docno, label - un df : docno, text
    queries = df[["qid", "query"]]
    corpus = df[["docno", "text"]]
    label = df[["qid", "docno", "label"]]

    return queries, corpus, label

queries_mantis, corpus_mantis, label_mantis = process_mantis(mantis_train)

In [8]:
corpus_mantis

,docno,text
0,d1,No idea. I think that thread had a bit of a di...
1,d2,Yes we both have iPhones. I think the sound qu...
2,d3,I believe 2 or 3 of the ones I listed above AL...
3,d4,It sounds like it's directly related to me. Ne...
4,d5,"""I have """"iTunes U"""" hidden under the General ..."
...,...,...
82267,d82268,"""@Century I just plugged *""""minimum viable pop..."
82268,d82269,I fixed the numbers which I had obviously gott...
82269,d82270,I am having trouble visualizing what you descr...
82270,d82271,"@N.Presley, I mentioned Cossacks, because your..."


In [9]:
queries_mantis

,qid,query
0,q1,"""I have a mid-2007 MacBook that, according to ..."
1,q2,The iPhone 3G supports Bluetooth stereo (A2DP)...
2,q3,"""I keep hearing all this hoopla about BBM (Bla..."
3,q4,I was a very happy customer of my microcell un...
4,q5,I’ve just started using iTunes U on my iPhone ...
...,...,...
82267,q82268,"""I'm working on creating a species that favour..."
82268,q82269,In my desert canal world all water is supplied...
82269,q82270,"""I'm in the process of designing a creature lo..."
82270,q82271,"""There is a walled town, situated at a river f..."


In [10]:
label_mantis

,qid,docno,label
0,q1,d1,1
1,q2,d2,1
2,q3,d3,1
3,q4,d4,1
4,q5,d5,1
...,...,...,...
82267,q82268,d82268,1
82268,q82269,d82269,1
82269,q82270,d82270,1
82270,q82271,d82271,1


In [11]:
def build_index(corpus, index_path="./index_mantis"):
    """
    créer l'index pour la recherche inversée à partir d'un corpus de documents clean
    corpus : (pd.DataFrame) : df contenant les documents
    """
    if os.path.exists(index_path) and os.path.exists(os.path.join(index_path, "data.properties")):
        print(f"Chargement de '{index_path}'")
        index = pt.IndexFactory.of(os.path.abspath(index_path))
    else:
        indexer = pt.IterDictIndexer("./index_mantis", fields=True)
        index_ref = indexer.index(
            corpus.to_dict(orient="records")
        )
        index = pt.IndexFactory.of(index_ref)
    return index

index_mantis = build_index(corpus_mantis)




Chargement de './index_mantis'


In [12]:

# pyterrier
def negative_sampling_mantis(method, qid, query, ground_truth, corpus, index, k=10):
    """
    pour une requête donnée et une méthode donnée (e.g. BM25), renvoie les k documents les plus pertinents, différents de la ground truth (le document réellement pertinent)
        Args:
            method (string) :  selection de la méthode (ex: "BM25", "TF_IDF", "random")
            request (String, String) : (qid, query) : possible en passant df.iloc[i]
            corpus (pd.DataFrame) : df contenant les documents à retrouver
            index (terrier.structures.Index) : index sur le corpus
        Returns:
            top_k_with_content (pd.DataFrame) : DataFrame avec les k documents et leur contenu
    """
    
    if method =="random":
        filtered_corpus = corpus[corpus["docno"] != ground_truth]
        
        # On tire k-1 documents au hasard
        random_samples = filtered_corpus.sample(n=k, random_state=0).copy()
        
        # On ajoute les colonnes qid et query pour garder la même structure que l'autre méthode
        random_samples["qid"] = qid
        random_samples["query"] = query
        
        return random_samples
        
    else:
        retriever = pt.terrier.Retriever(index, wmodel=method, num_results=k)

        safe_query = re.sub(r'[^\w\s]', ' ', query) # version clean uniquement pour BM25
        
        # df avec les colonnes qid, docid, docno, rank, score, query
        results_df = retriever.search(safe_query)

        results_df["query"] = query
        results_df["qid"] = qid
        
        # on retire le document correspondant à la ground_truth si il est présent
        filtered_df = results_df[results_df["docno"] != ground_truth]
        
        # si la ground_truth n'est pas là on garde les k premiers documents
        top_k_df = filtered_df.head(k).copy()

        # normalisation du score
        score_min = top_k_df["score"].min()
        score_max = top_k_df["score"].max()
        
        if score_max > score_min:
            top_k_df["score"] = (top_k_df["score"] - score_min) / (score_max - score_min)
        else:
            # Si tous les scores sont identiques -> distribution uniforme
            top_k_df["score"] = 1.0
        
        top_k_with_content = top_k_df.merge(corpus, on="docno", how="left")
        return top_k_with_content
    

In [13]:
qid, query = queries_mantis.iloc[0]
ground_truth = label_mantis[(label_mantis["qid"] == qid) & (label_mantis["label"] == 1)]["docno"].item()
negative_sampling_mantis("BM25", qid, query, ground_truth, corpus_mantis, index_mantis)

,qid,docid,docno,rank,score,query,text
0,q1,2827,d2828,0,1.000000,"""I have a mid-2007 MacBook that, according to ...","""Here is the link to [mac-forums](http://www.m..."
1,q1,6576,d6577,1,0.763938,"""I have a mid-2007 MacBook that, according to ...","Of course, it *should* be faster (just don't e..."
2,q1,35280,d35281,2,0.492955,"""I have a mid-2007 MacBook that, according to ...","Sure, So you'll have 5 volts coming in on the ..."
3,q1,3311,d3312,3,0.441492,"""I have a mid-2007 MacBook that, according to ...",That is difficult to answer. Perhaps this new ...
4,q1,2955,d2956,4,0.356167,"""I have a mid-2007 MacBook that, according to ...","Yes, the Mac Mini has a standard SATA port (ac..."
5,q1,20170,d20171,5,0.235431,"""I have a mid-2007 MacBook that, according to ...","As Learner said, the mentioned combo assure yo..."
6,q1,1057,d1058,6,0.135667,"""I have a mid-2007 MacBook that, according to ...",@gentmatt The Retina Macbook Pro uses an SSD m...
7,q1,4322,d4323,7,0.128757,"""I have a mid-2007 MacBook that, according to ...","""@WilliamD.Edwards *So this basically means yo..."
8,q1,19402,d19403,8,0.026633,"""I have a mid-2007 MacBook that, according to ...","It shouldn't be this complicated, but if nothi..."
9,q1,5224,d5225,9,0.000000,"""I have a mid-2007 MacBook that, according to ...","LOL, yeah that's the one we bought for our Can..."


In [14]:
def build_query_context(method, request, corpus, label, index, k=9):
    """
    Prépare les données pour une requête : k documents négatifs + 1 document positif (ground truth).
    
    Args:
        method (str) : "BM25", "TF_IDF" ou "random".
        request (tuple) : (qid, query).
        corpus (pd.DataFrame) : Le corpus (doit contenir "docno" et "text").
        label (pd.DataFrame) : La ground truth.
        index (pt.Index) : L'index PyTerrier.
        k (int) : Nombre d'exemples négatifs.
    
    Returns:
        pd.DataFrame : Les k+1 documents avec leur texte, score et label (1=pertinent, 0=non pertinent).
    """
    qid, query = request
    ground_truth = label[(label["qid"] == qid) & (label["label"] == 1)]["docno"].item()
    
    negative_sampling = negative_sampling_mantis(method, qid, query, ground_truth, corpus, index, k)
    negative_sampling.drop(columns=["rank", "docid"], errors="ignore", inplace=True)
    
    # pour la méthode random, on créer une colonne score rempli de -1
    if "score" not in negative_sampling.columns:
        negative_sampling["score"] = -1

    # pour le score on met directement 0.5 pour le label smoothing par la suite lorsqu'on utilisera le score BM25
    # pour la classe positive, on prend 1/K -> 0.5 dans le cas binaire
    row_ground_truth = pd.DataFrame(
        {
            "qid": [qid],
            "query": [query],
            "docno": [ground_truth], 
            "score": [0.5], 
            "text": [corpus[corpus["docno"]==ground_truth]["text"].item()]
        }
    )
    
    df = pd.concat([negative_sampling, row_ground_truth], ignore_index=True)
    
    df["label"] = (df["docno"] == ground_truth).astype(int)
    
    return df

In [15]:
build_query_context("BM25", queries_mantis.iloc[0], corpus_mantis, label_mantis, index_mantis)

,qid,docno,score,query,text,label
0,q1,d2828,1.000000,"""I have a mid-2007 MacBook that, according to ...","""Here is the link to [mac-forums](http://www.m...",0
1,q1,d6577,0.757479,"""I have a mid-2007 MacBook that, according to ...","Of course, it *should* be faster (just don't e...",0
2,q1,d35281,0.479081,"""I have a mid-2007 MacBook that, according to ...","Sure, So you'll have 5 volts coming in on the ...",0
3,q1,d3312,0.426211,"""I have a mid-2007 MacBook that, according to ...",That is difficult to answer. Perhaps this new ...,0
4,q1,d2956,0.338551,"""I have a mid-2007 MacBook that, according to ...","Yes, the Mac Mini has a standard SATA port (ac...",0
5,q1,d20171,0.214512,"""I have a mid-2007 MacBook that, according to ...","As Learner said, the mentioned combo assure yo...",0
6,q1,d1058,0.112018,"""I have a mid-2007 MacBook that, according to ...",@gentmatt The Retina Macbook Pro uses an SSD m...,0
7,q1,d4323,0.104919,"""I have a mid-2007 MacBook that, according to ...","""@WilliamD.Edwards *So this basically means yo...",0
8,q1,d19403,0.000000,"""I have a mid-2007 MacBook that, according to ...","It shouldn't be this complicated, but if nothi...",0
9,q1,d1,0.500000,"""I have a mid-2007 MacBook that, according to ...",No idea. I think that thread had a bit of a di...,1


In [16]:


def get_or_create_training_data(queries, corpus, labels, index, method="BM25", k=9, file_path="mantis_training_data.parquet"):
    """
    Charge le dataset s'il existe sur le disque, sinon le génère et le sauvegarde.
    """
    if os.path.exists(file_path):
        df_dataset = pd.read_parquet(file_path)
        print(f"Chargement terminé")
        return df_dataset
        
    print(f"Le fichier n'existe pas, generation")
    all_contexts = []
    
    for i in tqdm(range(len(queries))):
        request = queries.iloc[i]
        df_context = build_query_context(method, request, corpus, labels, index, k)
        all_contexts.append(df_context)
        
    df_dataset = pd.concat(all_contexts, ignore_index=True)
    
    print(f"Sauvegarde du dataset : {file_path}")
    df_dataset.to_parquet(file_path, index=False)
    
    return df_dataset


training_data_df = get_or_create_training_data(
    queries=queries_mantis, 
    corpus=corpus_mantis, 
    labels=label_mantis, 
    index=index_mantis, 
    method="BM25", 
    k=9,
    file_path="mantis_training_data.parquet"
)

Chargement terminé


In [17]:
training_data_df

,qid,docno,score,query,text,label
0,q1,d2828,1.000000,"""I have a mid-2007 MacBook that, according to ...","""Here is the link to [mac-forums](http://www.m...",0
1,q1,d6577,0.757479,"""I have a mid-2007 MacBook that, according to ...","Of course, it *should* be faster (just don't e...",0
2,q1,d35281,0.479081,"""I have a mid-2007 MacBook that, according to ...","Sure, So you'll have 5 volts coming in on the ...",0
3,q1,d3312,0.426211,"""I have a mid-2007 MacBook that, according to ...",That is difficult to answer. Perhaps this new ...,0
4,q1,d2956,0.338551,"""I have a mid-2007 MacBook that, according to ...","Yes, the Mac Mini has a standard SATA port (ac...",0
...,...,...,...,...,...,...
795509,q82272,d66848,0.185929,"""Could you map out all the connections in some...",@tryingToGetProgrammingStraight - Any *current...,0
795510,q82272,d31855,0.090150,"""Could you map out all the connections in some...","They were there at that price a year ago, and ...",0
795511,q82272,d81828,0.008335,"""Could you map out all the connections in some...",I'm saying that's the single most precise obje...,0
795512,q82272,d79472,0.000000,"""Could you map out all the connections in some...","""@Bato. I think our interpretations of `S` are...",0


In [18]:


def split_dataset_by_query(df, train_ratio=0.9, random_state=0, save_paths=None):
    """
    Sépare un dataset de Recherche d'Information en Train/Test en groupant par 'qid'.
    Garantit que tous les documents d'une même requête restent dans le même split.
    
    Args:
        df (pd.DataFrame): Le dataset complet contenant la colonne 'qid'.
        train_ratio (float): Proportion des requêtes allouées à l'entraînement (ex: 0.9).
        random_state (int): Graine aléatoire pour la reproductibilité du mélange.
        save_paths (tuple, optional): Tuple contenant (chemin_train, chemin_test) pour sauvegarder en Parquet.
        
    Returns:
        tuple: (train_df, test_df)
    """
    print(f"Dataset initial : {len(df)} lignes")
    
    #  mélange des requêtes 
    unique_qids = df["qid"].unique()
    np.random.seed(random_state)
    np.random.shuffle(unique_qids)
    
    split_index = int(len(unique_qids) * train_ratio)
    
    train_qids = unique_qids[:split_index]
    test_qids = unique_qids[split_index:]
    
    train_df = df[df["qid"].isin(train_qids)].copy()
    test_df = df[df["qid"].isin(test_qids)].copy()
    
    print(f"Split terminé :")
    print(f"  -> Train : {len(train_qids)} requêtes ({len(train_df)} lignes)")
    print(f"  -> Test  : {len(test_qids)} requêtes ({len(test_df)} lignes)")
    
    # sauvegarde optionnelle
    if save_paths and len(save_paths) == 2:
        train_path, test_path = save_paths
        train_df.to_parquet(train_path, index=False)
        test_df.to_parquet(test_path, index=False)
        print(f"Sauvegardé dans '{train_path}' et '{test_path}'")
        
    return train_df, test_df

In [19]:
train_df, test_df = split_dataset_by_query(
    df=training_data_df,
    train_ratio=0.9,
    random_state=0,
    save_paths=("mantis_train.parquet", "mantis_test.parquet")
)

Dataset initial : 795514 lignes
Split terminé :
  -> Train : 74044 requêtes (715874 lignes)
  -> Test  : 8228 requêtes (79640 lignes)
Sauvegardé dans 'mantis_train.parquet' et 'mantis_test.parquet'


# Label smoothing

## Label Smoothing classique
K correspondant au nombre de classes, i.e. K=2
le smoothing sur le label correspond à la formule suivante : **$(1 - \epsilon) \times \text{label\_origine} + \epsilon \times 0.5$**
avec epsilon arbitraire : les résultats présentés dans cet article sont avec un epsilon : 0.2 et 0.4

In [20]:
eps = 0.4

In [21]:
def label_smoothing_classique(sample_df, eps):
    if eps==0:
        return sample_df
    df = sample_df.copy()
    df["label"] = (1 - eps) * df["label"] + eps * 0.5
    return df

In [22]:
df_smoothed = label_smoothing_classique(training_data_df, eps)
print(df_smoothed[["qid", "score", "label"]].head(10))

  qid     score  label
0  q1  1.000000    0.2
1  q1  0.757479    0.2
2  q1  0.479081    0.2
3  q1  0.426211    0.2
4  q1  0.338551    0.2
5  q1  0.214512    0.2
6  q1  0.112018    0.2
7  q1  0.104919    0.2
8  q1  0.000000    0.2
9  q1  0.500000    0.8


## Weakly Supervised Label Smoothing

In [23]:
def ws_label_smoothing(sample_df, eps):
    if eps==0:
        return sample_df
    df = sample_df.copy()
    df["label"] = np.where(
        df["score"] != -1, 
        (1 - eps) * df["label"] + eps * df["score"], 
        df["label"] # Si le score est -1, on ne touche pas au label
    )
    return df
    


In [24]:
df_smoothed_ws = ws_label_smoothing(training_data_df, eps)
print(df_smoothed_ws[["qid", "score", "label"]].head(10))

  qid     score     label
0  q1  1.000000  0.400000
1  q1  0.757479  0.302991
2  q1  0.479081  0.191632
3  q1  0.426211  0.170484
4  q1  0.338551  0.135420
5  q1  0.214512  0.085805
6  q1  0.112018  0.044807
7  q1  0.104919  0.041968
8  q1  0.000000  0.000000
9  q1  0.500000  0.800000


## Dataset pytorch

In [31]:


class MantisDynamicDataset(Dataset):
    def __init__(self, training_data_df, tokenizer, max_len=128):
        self.data = training_data_df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        inputs = self.tokenizer(
            row["query"],
            row["text"],
            return_tensors="pt", 
            truncation=True,
            max_length=self.max_len, 
            padding='max_length',
            return_attention_mask=True
        )
        
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        
        # On renvoie les labels bruts les scores 
        #pour faire le smoothing dynamiquement lors de l'entrainement -> curriculum learning
        label = torch.tensor(row["label"], dtype=torch.float)
        score = torch.tensor(row["score"], dtype=torch.float)
        
        return input_ids, attention_mask, label, score




# instanciation
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

dataset_train = MantisDynamicDataset(train_df, tokenizer, max_len=256)#128)

# batch_size de 32
dataloader_train = DataLoader(
    dataset_train, 
    batch_size=32,#32, 
    shuffle=True, 
    num_workers=2
)

dataset_test = MantisDynamicDataset(test_df, tokenizer, max_len=256)#128)

dataloader_test = DataLoader(
    dataset_test, 
    batch_size=10, # 10 (car on a 1 vrai + 9 faux par requête)
    shuffle=False, # pas de shuffle pour garder les 10 docs ensembles
    num_workers=2
)



In [32]:
# test
batch_input_ids, batch_attention_mask, batch_labels, batch_scores = next(iter(dataloader_train))

print(f"Shape input_ids : {batch_input_ids.shape}")   # Devrait être [32, 128]
print(f"Shape labels : {batch_labels.shape}")         # Devrait être [32]
print(f"Shape scores : {batch_scores.shape}")         # Devrait être [32]
print(f"Exemple de labels : {batch_labels[:5]}")      # Ex: tensor([0., 0., 1., 0., 0.])
print(f"Exemple de scores : {batch_scores[:5]}")      # Ex: tensor([0.12, 0.45, 0.50, 0.89, 0.02])

Shape input_ids : torch.Size([32, 256])
Shape labels : torch.Size([32])
Shape scores : torch.Size([32])
Exemple de labels : tensor([0., 0., 0., 0., 0.])
Exemple de scores : tensor([0.0215, 1.0000, 0.5182, 0.1184, 0.2074])


In [33]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
model.to(device)



optimizer = AdamW(model.parameters(), lr=5e-6, eps=1e-8) 
loss_fn = BCEWithLogitsLoss()


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4963.35it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

In [34]:


def evaluate_r10_at_1(model, dataloader_test, device):
    """
    Évalue le modèle en calculant le R_10@1
    """
    model.eval()
    correct_predictions = 0
    total_queries = 0
    
    with torch.no_grad(): 
        for batch in dataloader_test:
            input_ids, attention_mask, labels, _ = [t.to(device) for t in batch]
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1) 
            
            best_doc_index = torch.argmax(logits).item()
            
            if labels[best_doc_index].item() == 1.0:
                correct_predictions += 1
                
            total_queries += 1
            
    r10_at_1 = correct_predictions / total_queries
    return r10_at_1

In [35]:
def train_t_wsls(model, dataloader_train, dataloader_test, optimizer, loss_fn, device, 
                 total_instances=50000, initial_eps=0.2, eval_every=10000):
    """
    Entraîne le modèle avec la méthode Two-stage Weakly Supervised Label Smoothing (T-WSLS).
    
    Args:
        model: Le modèle PyTorch.
        dataloader_train: Le DataLoader d'entraînement.
        dataloader_test: Le DataLoader de test pour l'évaluation.
        optimizer: L'optimiseur (ex: AdamW).
        loss_fn: La fonction de perte (ex: BCEWithLogitsLoss).
        device: 'cuda' ou 'cpu'.
        total_instances (int): Nombre total d'instances à voir.
        initial_eps (float): Valeur de l'epsilon pour la première moitié de l'entraînement.
        eval_every (int): Fréquence d'évaluation (en nombre d'instances).
        
    Returns:
        tuple: (history_steps, history_r10) pour tracer les courbes.
    """
    seen_instances = 0
    next_eval = eval_every
    
    history_steps = []
    history_r10 = []
    
    model.train()
    dataloader_iterator = iter(dataloader_train)

    if hasattr(tqdm, '_instances'):
        tqdm._instances.clear()
        
    progress_bar = tqdm(total=total_instances, desc="Entraînement T-WSLS")

    while seen_instances < total_instances:
        try:
            batch = next(dataloader_iterator)
        except StopIteration:
            dataloader_iterator = iter(dataloader_train)
            batch = next(dataloader_iterator)
            
        input_ids, attention_mask, labels, scores = [t.to(device) for t in batch]
        current_batch_size = input_ids.size(0)
        
        current_eps = initial_eps if seen_instances < (total_instances // 2) else 0.0
        
        smoothed_labels = torch.where(
            scores != -1,
            (1 - current_eps) * labels + current_eps * scores,
            labels
        )
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits.squeeze(-1), smoothed_labels) 
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        seen_instances += current_batch_size
        progress_bar.update(current_batch_size)
        progress_bar.set_postfix(loss=loss.item(), eps=f"{current_eps:.3f}")
        """        
        if seen_instances >= next_eval:
            tqdm.write(f"\n[Évaluation] {seen_instances}/{total_instances} instances")
            
            current_r10 = evaluate_r10_at_1(model, dataloader_test, device)
            tqdm.write(f"R10@1 actuel : {current_r10:.4f}")
            
            history_steps.append(seen_instances)
            history_r10.append(current_r10)
            
            model.train() 
            next_eval += eval_every
            """
    progress_bar.close()
    print("\nEntraînement terminé !")
    
    # Évaluation finale 
    final_r10 = evaluate_r10_at_1(model, dataloader_test, device)
    print(f"R10@1 FINAL : {final_r10:.4f}")
    #history_steps.append(seen_instances)
    #history_r10.append(final_r10)
    
    return final_r10#history_steps, history_r10

In [37]:


r10 = train_t_wsls(
    model=model,
    dataloader_train=dataloader_train,
    dataloader_test=dataloader_test,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    total_instances=50000,
    initial_eps=0.4,    
    eval_every=10000    
)

# Sauvegarde du modèle final
model.save_pretrained("./modele_mantis_twsls")
tokenizer.save_pretrained("./modele_mantis_twsls")


Entraînement T-WSLS: 50016it [12:06, 68.83it/s, eps=0.000, loss=0.245]                                                                          


Entraînement terminé !


R10@1 FINAL : 0.5049


Writing model shards: 100%|███████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.10s/it]


'\n# 3. Affichage des résultats\nplt.figure(figsize=(8, 5))\nplt.plot(history_steps, history_r10, marker=\'o\', linestyle=\'-\', color=\'b\')\nplt.title("Évolution du R10@1 pendant l\'entraînement (T-WSLS)")\nplt.xlabel("Instances d\'entraînement vues")\nplt.ylabel("R10@1")\nplt.grid(True, linestyle=\'--\', alpha=0.7)\nplt.show()'